# 🚀 ML Trader V10: Cloud Backtesting & WFO Hub
Welcome to the high-performance execution hub for `ml-trader-v10`. This notebook allows you to offload resource-intensive Walk-Forward Optimizations (WFO), hyperopts, and machine learning training to Google's cloud containers, saving your local MacBook Air from thermal throttling.

### ⚠️ Crucial First Step
Go to the menu above: **Runtime -> Change runtime type** and ensure that:
- **Hardware accelerator:** T4 GPU (free tier) or any active GPU/CPU.
- **Runtime shape:** Choose **High-RAM** if you are on Colab Pro (strongly recommended for 22 assets).

### 1. Mount Google Drive
This establishes a persistent link so that all trained FreqAI models, backtest logs, and data are saved permanently to your Google Drive and won't get deleted when Colab resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install TA-Lib C-Library and Wrapper
Compile and install the TA-Lib dependencies required for technical indicators in your strategies.

In [ ]:
%%bash
cd /content
wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
tar -xzf ta-lib-0.4.0-src.tar.gz
cd ta-lib
./configure --prefix=/usr > /dev/null
make > /dev/null
make install > /dev/null
cd ..
rm -rf ta-lib-0.4.0-src.tar.gz ta-lib
pip install TA-Lib -q

### 3. Clone Repository & Link Google Drive (Failsafe)
This pulls the latest strategy and code we developed locally into your Colab workspace and automatically links your Google Drive directories to prevent data and model loss.

In [ ]:
import os

# Safe directory reset to prevent deleted parent directory errors
%cd /content

# Clean up previous clone if any
!rm -rf /content/ml-trader-v10

# Clone the public repository
!git clone https://github.com/neel1104/ml-trader-v10.git /content/ml-trader-v10

%cd /content/ml-trader-v10

# Automatically link Google Drive models and data paths (failsafe!)
drive_models_path = '/content/drive/MyDrive/ml-trader-v10/models'
drive_data_path = '/content/drive/MyDrive/ml-trader-v10/data'
os.makedirs(drive_models_path, exist_ok=True)
os.makedirs(drive_data_path, exist_ok=True)

!rm -rf user_data/models && ln -s {drive_models_path} user_data/models
!rm -rf user_data/data && ln -s {drive_data_path} user_data/data

print("✓ Repository refreshed and linked to Google Drive successfully.")

### 4. Install Freqtrade and FreqAI/Hyperopt Requirements

In [ ]:
%%bash
if [ ! -d "/content/freqtrade" ]; then
    git clone https://github.com/freqtrade/freqtrade.git /content/freqtrade
    cd /content/freqtrade
    git checkout stable
fi
cd /content/freqtrade
pip install -e .[freqai,hyperopt] -q
pip install catboost statsmodels -q

### 5. Download Historical Candlestick Data
Fetch standard 5m, 15m, and 1h OHLCV data for your 22 assets. (Skip this cell if you have already synced your local data directory to your Google Drive!).

In [ ]:
# Downloads 250 days of historical Futures data from Binance (trading-mode futures is critical!)

In [ ]:
!python3 -m freqtrade download-data --config user_data/config.json --days 250 -t 5m 15m 1h --trading-mode futures

### 6. Run Walk-Forward Optimization (WFO)
Run the rolling out-of-sample backtests and generate the Walk-Forward Efficiency (WFE) score.

In [ ]:
# Execute the rolling WFO loop

In [ ]:
!python scripts/run_wfo.py --epochs 30

### 7. Custom Strategy Backtesting / Hyperopt
Run targeted backtests or tune specific hyperparameter spaces.

In [ ]:
# Target backtesting over a specific range

In [ ]:
!python3 -m freqtrade backtesting --config user_data/config.json --strategy StatArbStrategy --freqaimodel CatboostRegressor --timerange 20260401-20260425